<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.3-agent-engine/notebooks/GCP_Capstone_8.3_AgentEngine.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.3 Agent Engine: Managed Deployment, Memory Bank, Context, HIPAA
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q 'google-cloud-aiplatform[agent_engines,adk]>=1.112'
import os
os.environ['GOOGLE_CLOUD_PROJECT'] = 'documind-ai-YOUR-ID'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'  # global endpoint for Gemini 3.x generation
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print('Setup complete')


## Cell 1: Create Agent with Memory Bank Wiring


In [ ]:
from google.adk.agents import Agent
from google.adk.tools import ToolContext
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.adk.agents.callback_context import CallbackContext

# Memory write callback
async def save_memory(callback_context: CallbackContext):
    await callback_context.add_session_to_memory()
    return None

def search_documents(query: str, tool_context: ToolContext) -> dict:
    """Search documents.
    Args:
        query: Search query.
    """
    h = tool_context.state.get('search_history', [])
    h.append(query)
    tool_context.state['search_history'] = h
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report'}]}

def summarize_document(document_id: str, summary_type: str) -> dict:
    """Summarize a document.
    Args:
        document_id: Document ID.
        summary_type: brief/detailed/executive.
    """
    return {'summary': f'Summary of {document_id}'}

root_agent = Agent(
    name='documind',
    model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use past context when relevant.',
    tools=[PreloadMemoryTool(), search_documents, summarize_document],
    after_agent_callback=save_memory,
)
print('Agent with Memory Bank wiring created')


## Cell 2: Configure App with Context Caching + Compaction


In [ ]:
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.agents.context_cache_config import ContextCacheConfig

app = App(
    name='documind',
    root_agent=root_agent,
    context_cache_config=ContextCacheConfig(
        min_tokens=2048,
        ttl_seconds=600,
        cache_intervals=5,
    ),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,
        overlap_size=1,
    ),
)
print('App configured with caching + compaction')


## Cell 3: Deploy to Agent Engine


In [ ]:
import vertexai
from vertexai.agent_engines import AdkApp

client = vertexai.Client(
    project=os.environ['GOOGLE_CLOUD_PROJECT'],
    location='us-central1')  # Agent Engine deploys to a region; the agent's gemini-3.x model runs on global

adk_app = AdkApp(agent=root_agent, enable_tracing=True)

# Deploy (takes ~10 minutes)
# remote = client.agent_engines.create(
#     agent=adk_app,
#     config={
#         'display_name': 'DocuMind Production',
#         'requirements': ['google-cloud-aiplatform[agent_engines,adk]'],
#         'staging_bucket': 'gs://documind-staging',  # create first: gsutil mb -l us-central1 gs://documind-staging
#         'min_instances': 1,
#         'max_instances': 10,
#     }
# )
# print(f'Deployed: {remote.resource_name}')
print('Uncomment to deploy (requires GCP project with billing)')


## Cell 4: Query Deployed Agent


In [ ]:
# Uncomment when agent is deployed
# session = await remote.async_create_session(user_id='student1')
# async for event in remote.async_stream_query(
#     user_id='student1',
#     session_id=session['id'],
#     message='Find documents about revenue'
# ):
#     print(event)
#
# # Save session to Memory Bank
# await remote.async_add_session_to_memory(
#     user_id='student1',
#     session_id=session['id'])
# print('Session saved to Memory Bank')
print('Deploy first, then uncomment to query')


## Cell 5: HIPAA Compliance Checklist


In [ ]:
hipaa_checklist = {
    '1_baa': 'Execute BAA via Cloud Console > Privacy & Security > Legal',
    '2_encryption': 'AES-256 at rest (default) + CMEK + TLS 1.2+ in transit',
    '3_network': 'VPC Service Controls perimeter around Vertex AI resources',
    '4_access': 'IAM least-privilege + Cloud Audit Logs enabled',
    '5_models': 'Use only BAA-covered models (verify each model)',
    '6_region': 'Pin regional services (Firestore, embeddings, Document AI, Agent Engine) to your compliant region. NOTE: Gemini 3.x generation is served only from the global endpoint -- verify your BAA covers it, or use a region-pinned model for PHI.',
    '7_dlp': 'Cloud DLP scanning on data flows for PHI detection',
    '8_retention': 'Configure zero data retention at project level',
}

print('HIPAA Compliance Checklist for DocuMind:')
for k, v in hipaa_checklist.items():
    print(f'  [{k}] {v}')

print('\nIndia DPDP Act:')
print('  - Consent before processing personal data')
print('  - Data Principal rights (access/correction/erasure, 7-day window)')
print('  - 72-hour breach notification')
print('  - Use asia-south1 for Indian users')
print('  - Penalties: up to Rs.250 crore per violation')


## Cell 6: Cost Estimation


In [ ]:
# Agent Engine pricing
runtime_vcpu_hour = 0.0864
runtime_gib_hour = 0.0090
session_per_1k = 0.25
memory_store_per_1k = 0.25
memory_retrieve_per_1k = 0.50

# Gemini pricing
flash_input_1m = 1.50
flash_output_1m = 7.50
pro_input_1m = 2.00
pro_output_1m = 12.00

# DocuMind scenario: 1000 queries/month
# 80% Flash, 20% Pro. Avg 10K input tokens, 1K output tokens per query
flash_queries = 800
pro_queries = 200

flash_cost = flash_queries * (10000 * flash_input_1m / 1e6 + 1000 * flash_output_1m / 1e6)
pro_cost = pro_queries * (10000 * pro_input_1m / 1e6 + 1000 * pro_output_1m / 1e6)
runtime_cost = 2 * runtime_vcpu_hour * 730  # 2 vCPU, 730 hours/month
session_cost = 1000 * session_per_1k / 1000

total = flash_cost + pro_cost + runtime_cost + session_cost
cached_total = (flash_cost + pro_cost) * 0.15 + runtime_cost + session_cost  # caching discounts LLM input tokens, not infra

print(f'Monthly cost estimate (1000 queries):')
print(f'  Flash LLM: ${flash_cost:.2f}')
print(f'  Pro LLM: ${pro_cost:.2f}')
print(f'  Runtime: ${runtime_cost:.2f}')
print(f'  Sessions: ${session_cost:.2f}')
print(f'  TOTAL: ${total:.2f}')
print(f'  With context caching (~85% savings): ${cached_total:.2f}')


## Done!
- Agent Engine deployment (CLI + programmatic)
- Memory Bank wiring (PreloadMemoryTool + callback)
- Context caching + compaction
- HIPAA + DPDP Act compliance
- Cost estimation
